# Reinforcement Learning

In supervised learning, the algorithm learns from labeled examples. In unsupervised learning, it discovers structure in unlabeled data. **Reinforcement learning (RL)** takes a fundamentally different approach: an **agent** learns by **interacting** with an **environment**, receiving feedback in the form of **rewards** and **penalties**, and gradually discovering which actions lead to the best long-term outcomes.

There is no dataset of correct answers. Instead, the agent tries actions, observes consequences, and adjusts its strategy over time. A game-playing AI does not study labeled positions — it plays millions of games, learns from wins and losses, and improves. A robot does not memorize a map of every movement — it explores, receives reward for reaching goals, and learns a policy for navigation.

Reinforcement learning powers game-playing systems that surpass human champions, robots that learn locomotion, recommendation engines that optimize long-term engagement, and the alignment of large language models through human feedback. This notebook builds the conceptual foundation and demonstrates core algorithms from first principles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Reinforcement Learning Framework

Reinforcement learning involves four core components:

**Agent** — the learner and decision-maker. It chooses actions based on its current policy.

**Environment** — everything the agent interacts with. It receives actions, transitions to new states, and returns rewards.

**State** ($s$) — a description of the current situation. A chess board position, a robot's location and sensor readings, the current screen pixels in an Atari game.

**Action** ($a$) — what the agent can do. Move a chess piece, apply torque to a motor, press a joystick button.

**Reward** ($r$) — a scalar feedback signal. +1 for winning, -1 for losing, +0.01 for staying alive each timestep, -100 for crashing.

The interaction loop:

```
  ┌──────────────────────────────────────────────┐
  │                                              │
  │   Agent ──action──→ Environment              │
  │     ↑                    │                   │
  │     │                    │ state, reward      │
  │     └────────────────────┘                   │
  │                                              │
  └──────────────────────────────────────────────┘
```

At each timestep $t$, the agent observes state $s_t$, takes action $a_t$, receives reward $r_{t+1}$, and transitions to state $s_{t+1}$. The goal is to maximize **cumulative reward** over time — not just immediate reward, but the total accumulated through a sequence of decisions.

---

## 2. Markov Decision Processes

Most reinforcement learning problems are formalized as a **Markov Decision Process (MDP)**. An MDP is defined by:

- $\mathcal{S}$ — set of states
- $\mathcal{A}$ — set of actions
- $\mathcal{P}$ — transition probability: $P(s' | s, a)$ — probability of reaching state $s'$ given state $s$ and action $a$
- $\mathcal{R}$ — reward function: $R(s, a, s')$ — expected reward for transitioning from $s$ to $s'$ via action $a$
- $\gamma$ — discount factor, $0 \leq \gamma \leq 1$

The **Markov property** states that the future depends only on the current state and action, not on the history of previous states. The current state contains all information needed to make an optimal decision.

A **policy** $\pi$ is the agent's strategy — a mapping from states to actions:

- **Deterministic policy:** $a = \pi(s)$
- **Stochastic policy:** $\pi(a | s)$ — probability of taking action $a$ in state $s$

The agent's objective is to find the **optimal policy** $\pi^*$ that maximizes expected cumulative reward.

---

## 3. Returns and the Discount Factor

Cumulative reward (the **return**) at time $t$ is:

$$G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

The **discount factor** $\gamma$ controls how much the agent values future rewards relative to immediate ones:

- $\gamma = 0$ — only care about immediate reward (myopic).
- $\gamma = 1$ — all future rewards count equally (far-sighted, but may not converge if episodes are infinite).
- $\gamma = 0.99$ — common in practice; future rewards matter but are slightly discounted.

Discounting serves two purposes: it ensures the return is finite even in continuing (non-terminating) environments, and it reflects that uncertain distant rewards are less valuable than certain near-term ones.

The agent maximizes **expected return**: $J(\pi) = \mathbb{E}_\pi[G_t]$ — the average cumulative reward when following policy $\pi$.

In [ ]:
# Effect of discount factor on return
rewards = [0, 0, 0, 0, 0, 0, 0, 0, 0, 100]  # big reward at end
gammas = [0.0, 0.5, 0.9, 0.99, 1.0]

print(f"Reward sequence: {rewards}")
print(f"{'γ':>6s} {'Return':>10s}")
print("-" * 18)
for g in gammas:
    G = sum(g**k * rewards[k] for k in range(len(rewards)))
    print(f"{g:6.2f} {G:10.2f}")

---

## 4. Value Functions

Value functions estimate how good it is to be in a state or to take an action in a state, under a given policy.

### 4.1 State Value Function

The **state value function** $V^\pi(s)$ is the expected return when starting in state $s$ and following policy $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi\left[G_t \mid s_t = s\right]$$

### 4.2 Action Value Function (Q-Function)

The **action value function** $Q^\pi(s, a)$ is the expected return when starting in state $s$, taking action $a$, then following policy $\pi$:

$$Q^\pi(s, a) = \mathbb{E}_\pi\left[G_t \mid s_t = s, a_t = a\right]$$

The relationship between them:

$$V^\pi(s) = \sum_a \pi(a|s) \, Q^\pi(s, a)$$

### 4.3 Optimal Value Functions

The **optimal** value functions satisfy:

$$V^*(s) = \max_a Q^*(s, a)$$

$$Q^*(s, a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a') \mid s, a\right]$$

The optimal policy simply chooses the action with the highest Q-value in each state: $\pi^*(s) = \arg\max_a Q^*(s, a)$.

---

## 5. The Bellman Equations

The Bellman equations express value functions recursively — the value of a state equals the immediate reward plus the discounted value of the next state.

**Bellman expectation equation** (for a given policy $\pi$):

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[R(s,a,s') + \gamma V^\pi(s')\right]$$

**Bellman optimality equation** (for the optimal policy):

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[R(s,a,s') + \gamma V^*(s')\right]$$

These equations are the foundation of most RL algorithms. They say: the value of where you are equals what you get now plus the discounted value of where you end up. Dynamic programming methods (policy iteration, value iteration) solve these equations exactly when the MDP is known. Model-free methods (Q-learning, SARSA) learn value functions from experience without knowing transition probabilities.

---

## 6. Exploration vs Exploitation

The agent faces a fundamental dilemma: should it **exploit** what it already knows (choose the action with the highest estimated value) or **explore** unknown actions that might lead to better outcomes?

Pure exploitation gets stuck in local optima — if the agent never tries a different restaurant, it never discovers a better one. Pure exploration wastes time on actions known to be poor.

### Common Strategies

**Epsilon-greedy** — with probability $\varepsilon$, choose a random action (explore); with probability $1 - \varepsilon$, choose the best known action (exploit). $\varepsilon$ is often decayed over time.

**Softmax (Boltzmann) exploration** — choose actions proportional to $\exp(Q(s,a) / \tau)$, where temperature $\tau$ controls randomness.

**Upper Confidence Bound (UCB)** — choose the action with the highest upper bound on its value, balancing estimated value with uncertainty.

Getting the exploration-exploitation balance right is one of the central challenges in reinforcement learning.

In [ ]:
# Multi-armed bandit: exploration vs exploitation
true_rates = np.array([0.3, 0.7, 0.5, 0.4, 0.6])
n_arms = len(true_rates)
n_rounds = 1000

def run_bandit(epsilon, n_rounds=n_rounds):
    counts = np.zeros(n_arms)
    values = np.zeros(n_arms)
    rewards = []
    for t in range(n_rounds):
        if np.random.random() < epsilon:
            action = np.random.randint(n_arms)
        else:
            action = np.argmax(values)
        reward = 1 if np.random.random() < true_rates[action] else 0
        counts[action] += 1
        values[action] += (reward - values[action]) / counts[action]
        rewards.append(reward)
    return np.cumsum(rewards) / np.arange(1, n_rounds + 1)

plt.figure(figsize=(10, 5))
for eps, color, label in [(0, "red", "Pure exploit (ε=0)"),
                           (0.1, "blue", "ε-greedy (ε=0.1)"),
                           (0.3, "green", "ε-greedy (ε=0.3)"),
                           (1.0, "gray", "Pure explore (ε=1)")]:
    avg = run_bandit(eps)
    plt.plot(avg, color=color, label=label, linewidth=1.5)

plt.axhline(true_rates.max(), color="black", linestyle="--", label=f"Optimal ({true_rates.max()})")
plt.xlabel("Round")
plt.ylabel("Average reward")
plt.title("Multi-Armed Bandit: Exploration-Exploitation Tradeoff")
plt.legend()
plt.show()

---

## 7. Q-Learning

Q-learning is a model-free algorithm that learns the optimal action-value function $Q^*(s, a)$ directly from experience. The agent does not need to know transition probabilities — it learns by trying actions and observing results.

The **Q-learning update rule**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$

where:
- $\alpha$ is the **learning rate** (how much to update).
- $r$ is the immediate reward.
- $\gamma$ is the discount factor.
- $\max_{a'} Q(s', a')$ is the best estimated value of the next state.
- The term in brackets is the **TD error** (temporal difference error) — the difference between the new estimate and the old one.

Q-learning is **off-policy** — it learns the optimal Q-function while following any behavior policy (typically epsilon-greedy). The max over next actions means it always updates toward the best possible action, regardless of what action is actually taken next.

After sufficient exploration, the optimal policy is: $\pi(s) = \arg\max_a Q(s, a)$.

In [ ]:
# Q-Learning on a grid world
# Grid: S=start, G=goal, #=wall, .=open
# Agent can move up/down/left/right

GRID = [
    ['.', '.', '.', 'G'],
    ['.', '#', '.', '.'],
    ['.', '#', '.', '.'],
    ['S', '.', '.', '.'],
]
ROWS, COLS = len(GRID), len(GRID[0])
ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # up, down, left, right
ACTION_NAMES = ['↑', '↓', '←', '→']

def find_pos(char):
    for r in range(ROWS):
        for c in range(COLS):
            if GRID[r][c] == char:
                return (r, c)
    return None

start = find_pos('S')
goal = find_pos('G')

def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc
    if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or GRID[nr][nc] == '#':
        return state, -1, False  # hit wall, small penalty
    if (nr, nc) == goal:
        return (nr, nc), 10, True  # reached goal
    return (nr, nc), -0.1, False  # small step penalty

Q = defaultdict(lambda: np.zeros(4))
alpha, gamma, epsilon = 0.1, 0.99, 0.2
n_episodes = 500

for ep in range(n_episodes):
    state = start
    done = False
    while not done:
        if np.random.random() < epsilon:
            action = np.random.randint(4)
        else:
            action = np.argmax(Q[state])
        next_state, reward, done = step(state, action)
        Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
        state = next_state

policy_grid = [[' ' for _ in range(COLS)] for _ in range(ROWS)]
for r in range(ROWS):
    for c in range(COLS):
        if GRID[r][c] == '#':
            policy_grid[r][c] = '#'
        elif GRID[r][c] == 'G':
            policy_grid[r][c] = 'G'
        else:
            policy_grid[r][c] = ACTION_NAMES[np.argmax(Q[(r, c)])]

print("Learned policy (arrows show best action):")
for row in policy_grid:
    print('  ' + '  '.join(row))

In [ ]:
# Visualize Q-value convergence
Q_track = defaultdict(lambda: np.zeros(4))
episode_rewards = []

for ep in range(n_episodes):
    state = start
    total_reward = 0
    done = False
    while not done:
        action = np.random.randint(4) if np.random.random() < epsilon else np.argmax(Q_track[state])
        next_state, reward, done = step(state, action)
        Q_track[state][action] += alpha * (reward + gamma * np.max(Q_track[next_state]) - Q_track[state][action])
        total_reward += reward
        state = next_state
    episode_rewards.append(total_reward)

plt.figure(figsize=(10, 4))
window = 20
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
plt.plot(smoothed, linewidth=1.5)
plt.xlabel("Episode")
plt.ylabel(f"Reward (smoothed, window={window})")
plt.title("Q-Learning: reward per episode increases as policy improves")
plt.show()

---

## 8. SARSA and On-Policy Learning

**SARSA** (State-Action-Reward-State-Action) is an on-policy alternative to Q-learning. The update uses the action actually taken in the next state, not the maximum:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma Q(s', a') - Q(s, a)\right]$$

where $a'$ is the action chosen by the current policy in state $s'$.

| | Q-Learning | SARSA |
|---|---|---|
| Type | Off-policy | On-policy |
| Next action | $\max_{a'} Q(s', a')$ | $Q(s', a')$ where $a'$ is from current policy |
| Learns | Optimal policy | Policy being followed |
| Behavior | More aggressive | More conservative |

Q-learning learns the optimal policy regardless of exploration behavior. SARSA learns the value of the policy it is actually following — including exploration. In risky environments, SARSA may learn safer paths because it accounts for exploratory actions that could lead to danger.

---

## 9. Policy Gradient Methods

Value-based methods (Q-learning, SARSA) learn value functions and derive policies from them. **Policy gradient methods** directly optimize the policy parameters $\theta$ by gradient ascent on expected return:

$$\theta \leftarrow \theta + \alpha \, \nabla_\theta J(\theta)$$

The **policy gradient theorem** shows:

$$\nabla_\theta J(\theta) = \mathbb{E}_\pi \left[\nabla_\theta \log \pi_\theta(a|s) \cdot G_t \right]$$

Intuition: increase the probability of actions that led to high returns, decrease the probability of actions that led to low returns.

**REINFORCE** — the simplest policy gradient algorithm. Run full episodes, compute returns, and update policy parameters proportional to return times log-probability gradient.

**Advantages of policy gradients:**
- Can learn stochastic policies (important when optimal play requires randomness).
- Handle continuous action spaces naturally.
- Better convergence properties in some settings.

**Disadvantages:**
- High variance in gradient estimates.
- Sample inefficient — need many episodes.

---

## 10. Actor-Critic Methods

Actor-critic methods combine the strengths of value-based and policy-based approaches:

- **Actor** — the policy network, which selects actions.
- **Critic** — the value network, which evaluates how good the current state is.

The critic reduces variance in the policy gradient by providing a baseline. Instead of using raw return $G_t$, the actor updates using the **advantage**:

$$A(s, a) = Q(s, a) - V(s)$$

The advantage measures how much better action $a$ is compared to the average action in state $s$.

**A2C / A3C** (Advantage Actor-Critic / Asynchronous A3C) — parallel actor-critic with advantage estimation.

**PPO** (Proximal Policy Optimization) — constrains policy updates to prevent destructively large changes. Stable, sample-efficient, and widely used in robotics and game AI.

**DDPG / TD3 / SAC** — actor-critic methods for continuous action spaces, used in robot control.

---

## 11. Deep Reinforcement Learning

Traditional RL algorithms (Q-learning, SARSA) use tables to store Q-values. This works for small, discrete state spaces but fails when states are high-dimensional — images, continuous sensor readings, or large grids.

**Deep Reinforcement Learning** uses neural networks as function approximators to generalize across states.

### 11.1 Deep Q-Network (DQN)

DQN uses a neural network to approximate $Q(s, a)$. Key innovations that made it work:

- **Experience replay** — store transitions $(s, a, r, s')$ in a buffer and sample random mini-batches for training. Breaks correlation between consecutive samples and reuses past experience.
- **Target network** — a separate copy of the Q-network updated periodically. Stabilizes training by preventing the target from shifting every step.

DQN achieved human-level performance on many Atari 2600 games, learning directly from pixel inputs.

### 11.2 Extensions

- **Double DQN** — reduces overestimation of Q-values by decoupling action selection from evaluation.
- **Dueling DQN** — separates value and advantage streams in the network architecture.
- **Rainbow DQN** — combines multiple DQN improvements.

### 11.3 AlphaGo and Beyond

AlphaGo combined deep neural networks (policy and value networks) with Monte Carlo Tree Search (MCTS) to defeat the world champion at Go — a game with more possible positions than atoms in the universe. AlphaZero generalized this approach to chess and shogi with no human game data, learning entirely through self-play.

---

## 12. Reinforcement Learning from Human Feedback (RLHF)

RLHF applies reinforcement learning to align AI systems with human preferences. It has become central to training modern language models.

The three-stage process:

1. **Supervised fine-tuning** — train the model on high-quality human-written demonstrations.
2. **Reward model training** — collect human comparisons (response A is better than response B) and train a reward model to predict human preferences.
3. **RL optimization** — use the reward model as the reward signal and optimize the language model policy using PPO or similar algorithms.

The language model is the **agent**. Generating a response is an **action**. The reward model scores the response quality. The model learns to produce outputs that humans prefer — helpful, honest, harmless.

RLHF bridges the gap between predicting the next token (self-supervised pre-training) and producing behavior aligned with human values.

---

## 13. Applications of Reinforcement Learning

| Domain | Application | How RL Helps |
|--------|-------------|-------------|
| Games | Chess, Go, Atari, StarCraft | Learn optimal strategies through self-play |
| Robotics | Walking, grasping, navigation | Learn motor control through trial and error |
| Autonomous vehicles | Path planning, lane keeping | Optimize driving decisions in dynamic environments |
| Recommendation | Content ranking, ad placement | Maximize long-term user engagement, not just clicks |
| Finance | Portfolio management, trading | Optimize risk-adjusted returns over time |
| Operations | Data center cooling, traffic lights | Minimize energy cost, reduce congestion |
| NLP | RLHF for language models | Align model outputs with human preferences |
| Healthcare | Treatment planning, drug dosing | Personalize sequential treatment decisions |

---

## 14. Challenges in Reinforcement Learning

**Sample efficiency** — RL often requires millions of interactions. A robot cannot afford to fail thousands of times in the real world. Solutions: simulation, transfer learning, model-based RL.

**Sparse rewards** — in many problems, reward is rare (only at the end of a long episode). The agent receives no guidance for most steps. Solutions: reward shaping, curiosity-driven exploration, hierarchical RL.

**Credit assignment** — which actions in a long sequence were responsible for the final reward? A chess move 30 turns ago may have caused the win. Solutions: temporal difference learning, eligibility traces.

**Stability** — combining function approximation (neural networks) with bootstrapping (using own estimates as targets) can diverge. Solutions: experience replay, target networks, careful hyperparameter tuning.

**Safety** — exploratory actions can be dangerous in real-world settings (medical, autonomous driving). Solutions: constrained RL, safe exploration, human oversight.

**Non-stationarity** — the environment may change over time. A recommendation system faces shifting user preferences. Solutions: continual learning, adaptive policies.

---

## 15. When to Use Reinforcement Learning

Reinforcement learning is the right paradigm when:

- The problem involves **sequential decision-making** — current actions affect future states and rewards.
- There is a clear **reward signal** (or one can be designed).
- An **environment** can be simulated or accessed for trial and error.
- Long-term consequences matter — not just immediate outcomes.

Reinforcement learning is **not** the right choice when:

- Labeled data is available and the problem is a straightforward prediction task (use supervised learning).
- The goal is to discover structure in unlabeled data (use unsupervised learning).
- A single decision with no sequential consequences is needed.
- The environment cannot be simulated and real-world trial and error is too costly or dangerous.

In practice, RL is most successful when combined with simulation (games, robotics simulators), offline datasets (batch RL), or human feedback (RLHF).

In [ ]:
# Comparing RL algorithm families
algorithms = {
    "Q-Learning": {"type": "Value-based", "policy": "Derived from Q", "space": "Discrete", "sample_eff": "Low"},
    "SARSA": {"type": "Value-based", "policy": "Derived from Q", "space": "Discrete", "sample_eff": "Low"},
    "DQN": {"type": "Value-based", "policy": "Derived from Q", "space": "Discrete", "sample_eff": "Medium"},
    "REINFORCE": {"type": "Policy-based", "policy": "Direct", "space": "Both", "sample_eff": "Low"},
    "PPO": {"type": "Actor-Critic", "policy": "Direct", "space": "Both", "sample_eff": "Medium"},
    "SAC": {"type": "Actor-Critic", "policy": "Direct", "space": "Continuous", "sample_eff": "Medium"},
}

df_algo = pd.DataFrame(algorithms).T
print(df_algo.to_string())

---

## 16. Summary

Reinforcement learning trains agents to make sequential decisions by maximizing cumulative reward through environment interaction. The formal framework is the **Markov Decision Process**, with **policies**, **value functions**, and the **Bellman equations** providing the mathematical foundation.

**Q-learning** learns optimal action values from experience. **Policy gradient methods** optimize policies directly. **Actor-critic** methods combine both approaches. **Deep RL** scales these ideas to high-dimensional state spaces using neural networks. **RLHF** applies reinforcement learning to align AI systems with human preferences.

The exploration-exploitation tradeoff, sample efficiency, and credit assignment remain central challenges. Reinforcement learning excels when decisions unfold over time, feedback comes as rewards, and an environment is available for practice — from board games to robotics to language model alignment.